# Learning Objectives

After reading this notebook, students should be able to

+ Explain the algorithms in Dynamic Programming

+ Explain the concept of Generalized Policy Iteration

# Pre-requisites

+ Markov Decision Processes (MDP) and the Bellman equations associated with an MDP

# Dynamic Programming

The Bellman equations that we looked at in the last two chapters can be solved using matrix inversions and other naive methods, but these methods have a bad runtime complexity. These methods become infeasible when the MDP that we are working with is large. Therefore, we need a better way to evaluate the value functions. 

Also, a glaring limitation in what we have learned so far is that we know how to evaluate a policy, but we have no way to improve the policy. Improving the policy is the essence of RL, and we need to be able to do that. 

Dynamic Programming (DP) is a method that helps us out in both of these problems. First, it alleviates the complexity of solving Bellman equations by introducing iterative algorithms. Also, DP gives us steps to improve the policy iteratively. 



DP is a way to formulate a problem so that it can be broken down into individual steps that run in iterations. Each iteration brings us closer and closer to an answer. Each step uses the individual results from the previous steps using techniques such as caching. 

We need to reiterate an important point here: This chapter is about learning the best policy using a model of the environment. What we mean by this is that we know the $\text{Pr}(s'|s,a)$ function. Sometimes, we know this function and sometimes we don't. The next chapter introduces a way for us to learn a policy even when we do not know the environment dynamics. 

### Policy Evaluation
 
Solving the Bellman Equation(BE)s gives us the value function for a policy. This tells us how good a policy is. We can use a dynamic programming approach to solve the BE equations. Policy evaluation is a DP-based approach to evaluating the policy. The main idea in policy evaluation is to change the Bellman equation into an update rule
$$
v_{k+1}(s) = \sum _{a \in A} \pi(a|s) (R^a _s + \gamma \sum _{s' \in S}p^a _{ss'} v_k(s'))
$$
 
Let's take a grid world in which the goal is to get to the top-right state. As an example, we will evaluate a uniform policy, that is, in every state, we are equally likely to take all four actions. Let's start by randomly the value function to 0 for all the states.

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1-u3uan1EuvUA1LimsIoSoEgWB7ch6mW3" width="300">
     <figcaption>Figure 1: A gridworld with a goal state G which gives a reward of +1, and a state with a hole (H) that gives a reward of -1.</figcaption>
    </figure>
</div>

Next, let's look at the bellman equation for the value of the state 7 (state below the goal). The bellman equation for this state looks like this:
$$
V_\pi(S=s_7) = 0.25*(1+\gamma*V_\pi(S=s_3)) + 0.25*(0+\gamma*V_\pi(S=s_6)) + 0.25*(0+\gamma*V_\pi(S=s_{11})) + 0.25*(0+\gamma*V_\pi(S=s_7))
$$



Note that we have ommitted a bunch of terms considering that the actions taken deterministically causes the corresponding state transitions, except that actions that would take the agent off the grid and in such case such action leaves the state unchanged. That is, if the agent takes left action from state 7, it will end up in state 6 and if it takes action right from state 7, the state will remain unchanged and the agent will remain in state 7.


The above equation is a linear equation, it just tells us that the value of the state has a linear relation to the values of its neighbors. But what if we *evaluate* this equation? Let's do just that, by putting in the values of the states (neighbors) into the equation. 

$$
\begin{align}
V_\pi(S=s_7) &= 0.25 * [(1+\gamma*0) + (0+\gamma*0) + (0+\gamma*0) + (0+\gamma*0)]
\\
&= 0.25
\end{align}
$$

Let's repeat this for all the states in the grid. Because we do not assign the new values to the states until we have evaluated the value function for all the states in the grid, the order in which we visit the states does not matter. After an iteration of policy evaluation, the value function looks like this:

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1dSty4g0Brq9tc6o0mR3Z45lIOegcTjz4" width="300">
     <figcaption>Figure 2: State value function after an iteration of Policy Evaluation</figcaption>
    </figure>
</div>

We repeat this process until the value function converges. Since there is always one and only one unique solution to a set of linear equations, the algorithm converges to the actual solution. 

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1iVpeYPfLRWYlCx50dscJ0PEWAy2bv8_i" width="300">
     <figcaption>Figure 3: State value function after convergence of Policy Evaluation</figcaption>
    </figure>
</div>

Here is the policy evaluation algorithm.

> Algorithm parameters: Arbitrarily initialized value function $V_\pi$, A small threshold $\theta$, a policy $\pi$ that is to be evaluated
> 
>
> Loop:  
$\quad \Delta \leftarrow 0$   
$\quad$Loop  for  state $s$ in $S$:    
$\qquad v \leftarrow V(s)$   
$\qquad V(s) \leftarrow \sum_a \pi(s,a) \sum_{s', r} p(s', r | s, a) [r+\gamma V(s')]$  
$\qquad \Delta \leftarrow max(\Delta, |v-V(s)|)$  
$\quad$ until $\Delta < \theta$

## Policy Improvement

So far, we have established an algorithm to evaluate a policy so that we can see how good a policy is. We can also compare two policies by comparing their value functions, and choose the policy that is better than the other. But we cannot find the optimal policy by simply evaluating all the possible policies and selecting the one with the best value function--There are far too many possible policies. In fact, if you formulate the policy to be stochastic, an inifinite number of unique policies are possible. 

Let's take a policy which always selects the UP action in each of the state in our gridworld, and evaluate the value function for this policy using $\gamma=0.9$.

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1IkaIuCV31zDAgye9gpQZdxIKdvIENoK1" width="300">
     <figcaption>Figure 4: State value function for a policy that always takes the UP action in each state.</figcaption>
    </figure>
</div>

Remember that the value of an action in a state is your expected return if you take that action in the state, and then follow the policy from then on. For example, the action value of the action LEFT in the state 5 is the expected return when you take left at state 5, and then always UP from the rest of the states. 

Look at the state 6. The action value for the action RIGHT in state 6 is 0.9, which is greater than the state current value of the state (0). Pause for a bit and think of what this means. If we take RIGHT at state 6 instead of UP, we can get a higher return of 0.9 as compared to just 0 from going UP. If we change the policy so that we take LEFT at state 6 instead of UP, we can improve our policy. 

Policy improvement is an algorithm that does exactly this. It changes the policy so that we take the action that has the highest action value. Here is the policy improvement algorithm.

> For each $s \in S$:  
>> $\pi(s) \gets argmax_a \sum_{s^{'},r}P(s^´,r|s,a)[r+\gamma V^{\pi}(s^´) ]$  


> For each $s \in S$:  
> $\qquad \pi(s) \gets argmax_a \ q_\pi(s, a)$

## Policy Iteration

The policy changes in the policy improvement step. This means that the value function that we computed in policy iteration is no longer valid, and we can run policy iteration again to get the value function for this policy. If we repeat the two steps, that is evaluate and improve the policy in a loop, we get the policy iteration algorithm. Policy iteration gives us the optimal policy for an MDP. 
$$
\pi_0 \xrightarrow{\text{E}} v_{\pi_0} \xrightarrow{\text{I}} \pi_1 \xrightarrow{\text{E}} v_{\pi_1} \xrightarrow{\text{I}} \pi_2 \rightarrow \dots \xrightarrow{\text{I}} \pi_* \xrightarrow{\text{E}} v_{\pi_*} \xrightarrow{\text{I}} \pi_*
$$
where $\xrightarrow{\text{E}}$ denotes a policy evalution and $\xrightarrow{\text{I}}$ denotes a policy improvement.

Here is the policy iteration algorithm. 
>  1.Initialization  
>>    $V(s) \space \in R $ and $\pi(s) \space \in \space A(s)$ arbitarily for all $s \in S$

>   2.Policy Evaluation
>>  Loop:
>>> $\triangle \gets 0$  
>>> Loop for each $s \in S$
>>>> $v \gets V(s)$  
>>>> $V(s) \gets \Sigma_a \pi(a|s) \sum_{s^{'},r}P(s^´|s,a)[r+\gamma V^{\pi}(s^´) ]$  
>>>> $\triangle \gets max(\triangle,|v-V(s)|)$
    
>>until $\triangle < \theta$ (a small positive number determining  the accuracy of estimation)

>  3.Policy Improvement  
> policy-stable $\gets$ true  
> For each $s \in S$:  
>> old-action $\gets \pi(s)$  
>> $\pi(s) \gets argmax_a \sum_{s^{'},r}P(s^´,r|s,a)[r+\gamma V^{\pi}(s^´) ]$  
>> If $old-action \ne \pi(s)$,then $policy-stable \gets false$     

>If $policy-stable$, then stop and return $V \sim v_*$ and $\pi \sim \pi_*$ ; else goto 2



## Value iteration

The policy iteration algorithm completely separates the evaluation and improvement phases. The value function has to be fully computed before we can make any changes to the policy. Value iteration takes a more extreme approach, and does not wait for the full convergence of the value function before changing the policy. 

> Algorithm parameter: a small threshold $\theta \gt 0$ determining accuracy of estimation.   
> Initialize $V(s)$,for all $s \in S$, arbitrarily except that V(terminal)=0   
>Loop:  
>>$\triangle \gets 0$  
>>Loop for each $s \in S$:  
>>>$v \gets V(s)$  
>>>$V(s) \gets max_a\Sigma_{s´,r} P(s´,r|s,a)[r+\gamma V(s´)]$    
>>>$\triangle \gets max(\triangle,|v-V(s)|)$  

>untill $\triangle \lt \theta$   
> output a deterministic policy, $\pi \sim \pi_*$, such that   
>$\pi(s) = argmax_a \Sigma_{s´,r} P(s´,r|s,a)[r+\gamma V(s´)]$

## Asynchronous Dynamic Programming

The algorithms that we have looked at so far are *synchronous*. Each step happens one after the other in in a definite and perfect cadence. It turns out that we do not need to do everything in a synchronous manner. We can select any state at random, and perform a policy evaluation step for that state only. We can also run policy improvement on any single state at any point in time. This concept is called Asynchronous Dynamic Programming. 


Asynchronous DP converges to optimal policies and value functions as long as you select all the states in the MDP for evaluation and improvement. It can significantly reduce the required computation for finding an optimal policy. It also makes the algorithms simple to implement and parallelize. 

## Generalized Policy Iteration

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1b6BYHWhuEhmmCMldHrJJ89nBi7oPnba4" width="600">
     <figcaption>Figure 5: Optimal policies can be determined by iteratively calculating the value function for a policy, and improving the policy with respect to the value function. </figcaption>
    </figure>
</div>

Generalized Policy Iteration, or GPI for short, is a concept of letting the policy evaluation and improvement steps interact without considering the details of how or when the two processes occur. It generalizes the steps of policy evaluation and improvement, and abstracts away the specifics of these algorithms. GPI can be proven to have some useful convergence behaviour, which automatically proves the convergence behaviour for specific algorithms like Value iteration and Asynchronous DP. 

In each policy evaluation step, we update the value function with respect to the policy. In policy improvements, we improve the policy with respect to the value function. In each step, the value function and the policy grow more and more consistent with one another. At some point in this iteration, both processes stabilize and do not change anymore. We then consider that the value function and the policy have converged to some optimum. 

We can perform any number of sweep during evaluation, stop before it converge for a supplied policy and make policy improvement.Evaluation can be performed and stopped at any sweep. Value iteration is an example of GPI where we perform single sweep during policy evaluation.

In GPI ,evaluation and improvement processes are competing and cooperating to each other. They compete in the sense that they pull in opposing directions. 

Making the policy greedy with respect to the value function typically makes the value function incorrect for the changed policy, and making the value function consistent with the policy typically causes that policy no longer to be greedy. After running this interacting and competing process for longer iteration, these two processes interact to ﬁnd a single joint solution: the optimal value function and an optimal policy.

<div align="center">
    <figure>
     <img src="https://docs.google.com/uc?export=download&id=1gMPm6WmZH5XZCxqLorGuFeg-3nU7q0GM" width="400">
     <figcaption>Figure 6: The policy and the value function both set moving targets for each other, and converge together to the optimal policy and its value function. </figcaption>
    </figure>
</div>


As shown in this figure, the evaluation and improvement steps create a moving target for each other, and they both converge together to some optimum. 

# Key Takeaways

+ Dynamic programming is a family of algorithms for evaluating and improving policy using iterative methods. 

+ Using DP requires the complete model of the environment, which includes the transition probability function and the reward function.  

+ Policy evaluation iteratively evaluates the state value function of a policy.

+ Policy improvement is an  algorithm that selects the best action to take in a state by greedily selecting the action with the best action value.

+ Policy iteration alternates between policy evaluation and improvement to determine the optimal policy and the assiciated state value function.

+ Policy improvement alternates between a single sweep of policy evaluation and policy improvement to determine the optimal policy.

+ Asynchronous DP is the idea that we can visit the states and locally evaluate and update the policy in any order. Convergence is guaranteed as long as we visit all the states sufficient number of times.

+ Generalized policy iteration is the idea that the value function and the policy set moving targets for each other during the policy iteration, and they converge to the optimal values together.

## References

* Reinforcement Learning: An Introduction by *Richard S. Sutton* and *Andrew G. Barto*